# Build DF40

Crops DF40 ([YZY-stack/DF40](https://github.com/YZY-stack/DF40), CC BY-NC 4.0, non-commercial research only) with the lab's pipeline into a private dataset (`sources/df40.py`):

- the FF++-domain frames of the 31 known methods, for FF++ training and validation sequences only, one frame in `EVERY_FAKE`;
- the authors' crops of the FF++ real frames, every split (the test sequences are e4e's reals);
- the 9 unknown methods with the reals each ships, and e4e's inversions of FF++ test sequences.

The authors' frames are already tight face crops; RetinaFace finds the face again and the crop is cut at the lab's margin, so DF40 shares the other sources' framing. Where the authors' crop is tighter than that margin the lab crop is padded with black; the check below measures how much black is left inside the 1.3x framing the detectors are trained on.

The download links arrive by email after the access form and live in the Colab secret `DF40_LINKS`, a JSON object with the Drive ids `train_folder` (the folder the email calls "testing data", which holds the FF++-domain training frames), `test_folder` (the one it calls "training data", which holds the unknown methods) and `ffpp_real` (the FF++ real frames). Archives are downloaded, cropped and deleted one at a time; a rerun skips archives already cropped.

**Runtime:** Colab Pro, L4 or A100, several hours and about 30 GB of free disk at peak. **Needs** the Colab secrets `HF_TOKEN` (write access) and `DF40_LINKS`.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-df40"  # private
BUILD_DIR = "/content/build_df40"
ZIP_DIR = "/content/df40_zips"
EVERY_FAKE = 16  # one frame in 16 of each known method: about 2 per fake video

In [ ]:
%%capture
!pip install -q "face-decision-lab[build] @ {REPO_SOURCE}" gdown
!pip uninstall -y -q onnxruntime
!pip install -q onnxruntime-gpu

In [ ]:
import os
os.environ["DEEPFACE_BACKEND_ENGINE"] = "onnx"

import onnxruntime as ort
providers = ort.get_available_providers()
print("ONNX Runtime providers:", providers)
assert "CUDAExecutionProvider" in providers, "onnxruntime-gpu cannot see the GPU; RetinaFace would run on the CPU"

### Archives

In [ ]:
import json, re, html
from pathlib import Path
import requests
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))
links = json.loads(userdata.get("DF40_LINKS"))

def folder_files(folder_id):
    page = requests.get(f"https://drive.google.com/embeddedfolderview?id={folder_id}", timeout = 60).text
    entries = re.findall(r'file/d/([^/]+)/view[^>]*>.*?<div class="flip-entry-title">([^<]+)</div>', page, re.S)
    return {html.unescape(title).removesuffix(".zip"): file_id for file_id, title in entries}

train_archives, test_archives = folder_files(links["train_folder"]), folder_files(links["test_folder"])
print(len(train_archives), "training archives,", len(test_archives), "test archives")

### Crop

In [ ]:
import subprocess, zipfile
from tqdm.auto import tqdm
from face_decision_lab.build import DatasetBuilder
from face_decision_lab.faces import RetinaFaceDetector
from face_decision_lab.sources import df40

builder = DatasetBuilder(BUILD_DIR, RetinaFaceDetector())
Path(ZIP_DIR).mkdir(parents = True, exist_ok = True)

def download(file_id, path):
    url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"
    subprocess.run(["curl", "-sSL", "--retry", "10", "-C", "-", "-o", str(path), url], check = True)
    if not zipfile.is_zipfile(path):
        raise RuntimeError(f"{path} is not a zip archive; Drive may have refused the download (quota)")

def crop_archive(name, file_id, parse, keep):
    marker = Path(ZIP_DIR) / f"{name}.done"
    if marker.exists():
        return
    path = Path(ZIP_DIR) / f"{name}.zip"
    download(file_id, path)
    with zipfile.ZipFile(path) as archive:
        items = ((item, archive.read(member)) for member in archive.namelist()
                 if (item := parse(member)) is not None and keep(item))
        added = builder.add_images(items)
    builder.flush()
    path.unlink()
    marker.touch()
    print(f"{name}: {added} images")

for name in tqdm(sorted(df40.KNOWN_METHODS), desc = "known methods"):
    crop_archive(f"train_{name}", train_archives[name], lambda m, a = name: df40.parse_train_member(a, m),
                 lambda item: df40.official_split(item) in ("train", "val") and df40.sampled(item.path, EVERY_FAKE))
crop_archive("ffpp_real", links["ffpp_real"], df40.parse_real_member, lambda item: True)
for name in tqdm(sorted(df40.UNKNOWN_METHODS), desc = "unknown methods"):
    crop_archive(f"test_{name}", test_archives[name], lambda m, a = name: df40.parse_unknown_member(a, m),
                 lambda item: not item.subject_id.startswith("ffpp_") or df40.official_split(item) == "test")
print(len(builder.samples), "crops")

### Check

Detection outcome per method, and the share of near-black pixels inside the 1.3x framing (`tighten_crop`) of up to 200 crops per method: black there is padding added where the authors' crop was tighter than the lab's margin, and differs between reals and fakes only if their crops do.

In [ ]:
import numpy as np
from PIL import Image
from face_decision_lab.faces import tighten_crop

for generator in sorted(builder.report.counts):
    print(f"{generator:28s} saved {builder.report.saved_share(generator):6.1%}  {builder.report.counts[generator]}")
by_generator = {}
for sample in builder.samples:
    by_generator.setdefault(sample.generator, []).append(sample)
for generator, part in sorted(by_generator.items()):
    shares = [(np.asarray(tighten_crop(Image.open(Path(BUILD_DIR) / s.image_path).convert("RGB"), 1.3)).max(axis = -1) < 8).mean()
              for s in part[:200]]
    print(f"{generator:28s} crops {len(part):6d}  black inside 1.3x {np.mean(shares):6.2%}")

### Upload

In [ ]:
from face_decision_lab.hub import push_build

print(push_build(BUILD_DIR, HF_DATASET))